## Read in Data and Feature Engineering Steps

In [0]:
%run ./03_feature_engineering

In [0]:
# Set a random seed
RANDOM_SEED = 42

## Modeling Functions

In [0]:
import pandas as pd

def cv_evaluator(cv_model):
    # Create a list of dictionaries containing parameters and their scores
    results = []
    for params, metric in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics):
        row = {p.name: v for p, v in params.items()}
        row['score'] = metric
        results.append(row)

    # Display as a table
    df_results = pd.DataFrame(results)
    display(df_results.sort_values("score", ascending=False))

In [0]:
def coefficient_feature_extractor(cv_model, vectorAssembler):
    # Get coefficients (and corresponding feature names from vector assembler) and intercept of the best logistic regression model
    lr_model = cv_model.bestModel.stages[-1]
    feature_names = vectorAssembler.getInputCols()
    coefficients = lr_model.coefficients.toArray()
    intercept = lr_model.intercept
    
    # Print the coefficients/feature names and intercept, sort coeffiient values by absolute value
    for feature, coefficient in zip(feature_names, coefficients):
        sorted_coefficients = sorted(zip(feature_names, coefficients), key=lambda x: abs(x[1]), reverse=True)
    for feature, coefficient in sorted_coefficients:
        print(f"{feature}: {coefficient}")
    print(f"Intercept: {intercept}")

In [0]:
def rf_feature_importances(rf_cv_model, vectorAssembler):
    # Get feature importances of the best random forest model
    rf_model = rf_cv_model.bestModel
    feature_importances = rf_model.featureImportances.toArray()
    feature_names = vectorAssembler.getInputCols()
    
    # Extract feature names and feature importances
    feature_importances_df = pd.DataFrame({"Feature": feature_names, "Importance": feature_importances})
    
    # Sort feature importances in descending order
    feature_importances_df = feature_importances_df.sort_values(by="Importance", ascending=False)
    
    # Print feature importances
    for feature, importance in zip(feature_importances_df["Feature"], feature_importances_df["Importance"]):
        print(f"{feature}: {importance}")

## Churn Propensity Models

#### Configuration

In [0]:
%run ./_config

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.ml.feature import StandardScaler

#### Logistic Regression

In [0]:
from pyspark.ml.classification import LogisticRegression

_Note: weighted classes were examined but were not useful_

In [0]:
# # Build baseline model (no tuning)
# scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
# lr = LogisticRegression(featuresCol="scaled_features", labelCol="Churn")
# pipeline = Pipeline(stages=[scaler, lr])

# # Fit and train model
# lr_model = pipeline.fit(lr_all_train)

# # Make predictions on the test data
# lr_predictions = lr_model.transform(lr_all_test)

# # Evaluate the predictions
# evaluator = BinaryClassificationEvaluator(labelCol="Churn", metricName="areaUnderPR")
# print(f"Test Area Under PR: {evaluator.evaluate(lr_predictions)}")

# # Delete existing cached models
# try:
#     del lr_model
# except Exception as e:
#     print(f"Error deleting model: {e}")

- Baseline score (minimal feature engineering and tuning): =~ 0.3171
- All features (no tuning): =~ 0.3156

In [0]:
# # Define the scaler, model, and pipeline
# lr_scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
# lr = LogisticRegression(featuresCol="scaled_features", labelCol="Churn")
# lr_pipeline = Pipeline(stages=[lr_scaler, lr])

# # Define the parameter grid (reduced to avoid model size overflow)
# lr_param_grid = ParamGridBuilder() \
#     .addGrid(lr.regParam, [0.0, 1e-6, 1e-2, 0.05, 0.5]) \
#     .addGrid(lr.elasticNetParam, [1]) \
#     .build()

# # Define the evaluator and cross-validator
# lr_evaluator = BinaryClassificationEvaluator(labelCol="Churn", metricName="areaUnderPR")
# lr_cv = CrossValidator(estimator=lr_pipeline, evaluator=lr_evaluator, estimatorParamMaps=lr_param_grid, seed=RANDOM_SEED)

# # Train and tune the model
# lr_cv_model = lr_cv.fit(lr_all_train)

In [0]:
# # Examine cross-validation results
# cv_evaluator(lr_cv_model)

In [0]:
# # Examine feature coefficients and intercept values
# coefficient_feature_extractor(lr_cv_model, lr_all_assembler)

**_Feature selection with logistic regression model (elasticNetParam = 1, L1/lasso penalty; tuning regParam)_**

**Best lr_train parameters:** regParam: 0;	score: 0.31188806931714813

Coefficients (log odds):
- Age: 0.3041328646789718
- Age_sq: -0.247902159804467
- Discount_Usage: -0.06328175393323537
- Membership_Years: -0.063106984925925
- Online_Purchases: 0.023049345857847826
- Spending_Score: -0.012238040577599554
- Annual_Income: 0.00954058761576695
- Intercept: -0.7937009209868396

**Best lr_all_train parameters:** regParam: 0.0001;	score: 0.3278453439603538

Coefficients (log odds):
- Male: -0.45186013270329195
- Age: 0.33406513565085677
- Age_sq: -0.2684608509273
- Male_Discount_Usage: 0.2673594080593565
- Male_Spending_Score: 0.19587567270133713
- Male_Annual_Income: 0.14815445168360256
- Discount_Usage_Annual_Income: -0.13286458629264333
- Spending_Score: -0.1005862384461141
- Discount_Usage: -0.08762076644427258
- Membership_Years_Annual_Income: -0.08561605813905407
- Annual_Income: 0.07736691878903934
- Age_Membership_Years: -0.01851427325024529
- Male_Age: -0.016800923139428717
- Online_Purchases_Annual_Income: 0.016459105451085712
- Discount_Usage_Online_Purchases: 0.0070113736018968505
- Intercept: -0.7991565648472586

In [0]:
# # Delete existing cached models
# del lr_param_grid
# del lr_cv_model

In [0]:
from pyspark.ml.feature import VectorAssembler

# Narrow down features
lr_features_final = ["Male", "Age_sq", "Age", "Male_Discount_Usage", "Male_Spending_Score", "Male_Annual_Income", "Discount_Usage_Annual_Income"]
lr_final_assembler = VectorAssembler(inputCols=lr_features_final, outputCol="features")
lr_final_train = lr_final_assembler.transform(train)
lr_final_test = lr_final_assembler.transform(test)

lr_final_train = lr_final_train.select("features", "Churn")
lr_final_test = lr_final_test.select("features", "Churn")

In [0]:
# Define the scaler, model, and pipeline
lr_scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
lr = LogisticRegression(featuresCol="scaled_features", labelCol="Churn")
lr_pipeline = Pipeline(stages=[lr_scaler, lr])

# Define the parameter grid (reduced to avoid model size overflow)
lr_param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [1e-6, 1e-2, 0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.25]) \
    .build()

# Define the evaluator and cross-validator
lr_evaluator = BinaryClassificationEvaluator(labelCol="Churn", metricName="areaUnderPR")
lr_cv = CrossValidator(estimator=lr_pipeline, evaluator=lr_evaluator, estimatorParamMaps=lr_param_grid, seed=RANDOM_SEED)

# Train and tune the model
lr_cv_model = lr_cv.fit(lr_final_train)

In [0]:
# Examine cross-validation results
cv_evaluator(lr_cv_model)

In [0]:
# Examine feature coefficients and intercept values
coefficient_feature_extractor(lr_cv_model, lr_final_assembler)

**_Final model parameters:_** regParam: 0.000001	elasticNetParam: 0	score: 0.3352536916375981

Coefficients (log odds):
- Male: -0.3892212651350518
- Age: 0.28979326124995874
- Age_sq: -0.23981463596915734
- Male_Annual_Income: 0.21748009282813272
- Male_Discount_Usage: 0.20084190049773756
- Discount_Usage_Annual_Income: -0.16799698265706556
- Male_Spending_Score: 0.08050179340412492
- Intercept: -0.7963229072427156

In [0]:
# Make predictions on the test data
lr_predictions = lr_cv_model.transform(lr_final_test)

# Evaluate the predictions
evaluator = BinaryClassificationEvaluator(metricName="areaUnderPR", labelCol="Churn")
print(f"Test Area Under PR: {evaluator.evaluate(lr_predictions)}")

In [0]:
# Delete existing cached models
try:
    del lr_param_grid
    del lr_cv_model
except Exception as e:
    print(e)

#### Random forest

In [0]:
from pyspark.ml.classification import RandomForestClassifier

_Note: Although random forest models can usually identify interaction terms with deeper trees, manually-created feature terms are also examined due to maxDepth restrictions. Additionally, weights were examined but were not useful._

In [0]:
# # Build baseline model (no tuning)
# rf = RandomForestClassifier(labelCol="Churn", featuresCol="features", seed=RANDOM_SEED)

# # Train the model
# rf_model = rf.fit(train_data_all)

# # Make predictions on the test data
# rf_predictions = rf_model.transform(test_data_all)

# # Evaluate the predictions
# evaluator = BinaryClassificationEvaluator(metricName="areaUnderPR", labelCol="Churn")
# print(f"Test Area Under PR: {evaluator.evaluate(rf_predictions)}")

# # Delete existing cached models
# try:
#     del rf_model
# except Exception as e:
#     print(e)

- Baseline (no feature engineering, no tuning): =~ 0.3134
- All features (no tuning): =~ 0.2838

In [0]:
# # Initialize the classifier
# rf = RandomForestClassifier(labelCol="Churn", featuresCol="features", seed=RANDOM_SEED)

# # Define the parameter grid 
# rf_param_grid = ParamGridBuilder() \
#     .addGrid(rf.bootstrap, [True]) \
#     .addGrid(rf.subsamplingRate, [0.8]) \
#     .addGrid(rf.numTrees, [20, 40, 100]) \
#     .addGrid(rf.maxDepth, [15, 30]) \
#     .addGrid(rf.featureSubsetStrategy, ["sqrt"]) \
#     .addGrid(rf.minInstancesPerNode, [3]) \
#     .build()

# # Define the evaluator and cross-validator
# rf_evaluator = BinaryClassificationEvaluator(metricName="areaUnderPR", labelCol="Churn")
# cv_rf = CrossValidator(estimator=rf, evaluator=rf_evaluator, estimatorParamMaps=rf_param_grid, seed=RANDOM_SEED)

# # Train the model
# rf_cv_model = cv_rf.fit(train_data_all)

In [0]:
# cv_evaluator(rf_cv_model)

In [0]:
# rf_feature_importances(rf_cv_model, all_assembler)

**_Feature selection/importance random forest model_**

**Best train_data parameters:** numTrees: 40, maxDepth: 30, score: 0.30703393713488497

Feature importances:
- Spending_Score: 0.1689237883890488
- Age: 0.16595716970579064
- Discount_Usage: 0.16495046905729974
- Online_Purchases: 0.16104986531954715
- Membership_Years: 0.15941230142830498
- Annual_Income: 0.151461401334525
- Male: 0.02824500476548375

**Best train_data_all parameters:** numTrees: 20, maxDepth: 30, score: 0.3140912074416725

Feature importances:
- Age: 0.10060954249638732
- Membership_Years: 0.08523973498029559
- Age_Membership_Years: 0.07825235799244147
- Spending_Score: 0.07794822556869209
- Discount_Usage_Online_Purchases: 0.07794608600789488
- Annual_Income: 0.07735227848901581
- Discount_Usage: 0.07626986605865878
- Online_Purchases: 0.07403524146573968
- Membership_Years_Annual_Income: 0.06982225565521341
- Online_Purchases_Annual_Income: 0.06766012500045748
- Discount_Usage_Annual_Income: 0.06710938949599048
- Male_Spending_Score: 0.04076828174820423
- Male_Age: 0.036069017527498505
- Male_Discount_Usage: 0.0357270848364043
- Male_Annual_Income: 0.035190512677105844

In [0]:
# # Delete existing cached models
# try:
#     del rf_cv_model
#     del rf_param_grid
# except Exception as e:
#     print(e)

In [0]:
# Narrow down features
rf_features_final = ["Age", "Membership_Years", "Age_Membership_Years", "Spending_Score", "Discount_Usage_Online_Purchases", "Annual_Income", "Discount_Usage", "Online_Purchases"]
rf_final_assembler = VectorAssembler(inputCols=rf_features_final, outputCol="features")
rf_final_train = rf_final_assembler.transform(train)
rf_final_test = rf_final_assembler.transform(test)

rf_final_train = rf_final_train.select("features", "Churn")
rf_final_test = rf_final_test.select("features", "Churn")

In [0]:
# Initialize the classifier
rf = RandomForestClassifier(labelCol="Churn", featuresCol="features", seed=RANDOM_SEED)

# Define the parameter grid 
rf_param_grid = ParamGridBuilder() \
    .addGrid(rf.bootstrap, [True]) \
    .addGrid(rf.subsamplingRate, [0.8]) \
    .addGrid(rf.numTrees, [20, 25]) \
    .addGrid(rf.maxDepth, [30]) \
    .addGrid(rf.featureSubsetStrategy, ["sqrt"]) \
    .addGrid(rf.minInstancesPerNode, [3]) \
    .build()

# Define the evaluator and cross-validator
rf_evaluator = BinaryClassificationEvaluator(metricName="areaUnderPR", labelCol="Churn")
cv_rf = CrossValidator(estimator=rf, evaluator=rf_evaluator, estimatorParamMaps=rf_param_grid, seed=RANDOM_SEED)

# Train the model
rf_cv_model = cv_rf.fit(rf_final_train)

In [0]:
# Examine cross-validator results
cv_evaluator(rf_cv_model)

In [0]:
# Examine feature importances
rf_feature_importances(rf_cv_model, rf_final_assembler)

**_Final model parameters:_** numTrees: 20, maxDepth: 30, score: 0.3104824818211328

Feature importances:
- Spending_Score: 0.14750813006838387
- Age: 0.14350536571564915
- Membership_Years: 0.13565386048194333
- Discount_Usage: 0.12447242885822843
- Annual_Income: 0.12078456321296696
- Discount_Usage_Online_Purchases: 0.1164127657549318
- Online_Purchases: 0.11489900234215156
- Age_Membership_Years: 0.09676388356574486

In [0]:
# Make predictions on test set
rf_final_predictions = rf_cv_model.transform(rf_final_test)

# Evaluate predictions
rf_final_evaluator = BinaryClassificationEvaluator(metricName="areaUnderPR", labelCol="Churn")
rf_final_evaluator.evaluate(rf_final_predictions)

In [0]:
# Delete existing cached models
try:
    del rf_cv_model
    del rf_param_grid
except Exception as e:
    print(e)

#### XGBoost

#### Neural network